# Prices and volume over time

Reads `marts.resale_features`. Figures are saved to `reports/figures/` as PNG and SVG.

Before March 2012 sales are dated by approval date, afterwards by registration date, which is typically a few weeks later; the change is marked on both charts.

In [ ]:
import sys
from pathlib import Path

# Make the repository importable when the kernel starts in notebooks/.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

In [ ]:
from config.settings import get_settings
from pipeline.common.db import get_engine
from pipeline.eda import data, figures

figures.apply_style()
engine = get_engine()
FIGURES = get_settings().reports_dir / "figures"

## Median price per sqm by flat type

In [ ]:
prices = data.median_psm_by_month(engine)
fig = figures.price_by_flat_type(prices)
figures.save_figure(fig, "price_psm_by_flat_type", FIGURES)

In [ ]:
yearly = (
    prices.assign(year=prices["month"].dt.year)
    .groupby(["year", "flat_type"])["median_psm"]
    .median()
    .unstack()
)
yearly.loc[[1990, 1997, 2003, 2007, 2013, 2017, 2020, 2024, 2026]].round(0)

## Transaction volume by month

In [ ]:
volume = data.monthly_volume(engine)
fig = figures.monthly_volume(volume)
figures.save_figure(fig, "monthly_volume", FIGURES)

In [ ]:
by_year = volume.assign(year=volume["month"].dt.year).groupby("year")["sales"].sum()
lowest = volume.nsmallest(3, "sales")
by_year.to_frame().T, lowest